# Leaf stomatal profit-max
 

In [ ]:
# | default_exp leaf_stomatal_profit_max

In [ ]:
# | hide
from fastcore import *
from nbdev.showdoc import *

In [ ]:
# | export
import numpy as np
from plant_hydraulics.leaf_temperature import leaf_temperature
from plant_hydraulics.leaf_boundary_layer import leaf_boundary_layer
from plant_hydraulics.leaf_photosynthesis import leaf_photosynthesis
from plant_hydraulics.leaf_water_potential import leaf_water_potential
from plant_hydraulics.parameter_classes import PhysCon, Leaf, Flux, Atmos

In [ ]:
# | export


def _hydraulic_cost(psi_leaf, psi_50, a_psi):
    """
    Normalised hydraulic cost: fractional loss of conductance.

    Uses a sigmoidal vulnerability curve:

        cost(ψ) = 1 - 1 / (1 + (ψ / ψ_50)^a)

    where ψ_50 is the water potential at 50% loss of conductance
    (we use leaf.minl_wp as a proxy for P50), and a controls steepness.

    Behavior:
       - ψ = 0        → cost ≈ 0   (fully hydrated, no risk)
       - ψ = ψ_50     → cost = 0.5 (half conductance gone)
       - ψ << ψ_50    → cost → 1.0 (near-total loss)

    Example with ψ_50 = -2.0 MPa, a = 4:
        - ψ = -0.5 → cost = 0.004  (almost no risk)
        - ψ = -1.0 → cost = 0.059  (minor risk)
        - ψ = -1.5 → cost = 0.220  (moderate risk — stomata should start closing)
        - ψ = -2.0 → cost = 0.500  (50% conductance lost)
        - ψ = -2.5 → cost = 0.745  (severe — approaching lethal)

    This is equivalent to the Weibull-type vulnerability curves?

    Parameters
    ----------
    psi_leaf : float
        Current leaf water potential (MPa). Always ≤ 0.
    psi_50 : float
        Water potential at 50% loss of conductance (MPa). Always < 0.

    a_psi : float
        Shape parameter. 2-3 = gentle (isohydric), 5-8 = steep (anisohydric).

    Returns
    -------
    cost : float
        Fractional loss of conductance, 0 to 1.
    """
    if psi_50 == 0:
        return 0.0

    # Both psi_leaf and psi_50 are negative, so ratio is positive.
    # When psi_leaf is more negative than psi_50, ratio > 1, cost > 0.5
    ratio = psi_leaf / psi_50
    return 1.0 - 1.0 / (1.0 + ratio**a_psi)

In [ ]:
# | export


def leaf_stomatal_profit_max(
    physcon: PhysCon, atmos: Atmos, leaf: Leaf, flux: Flux
) -> Flux:
    """
    Stomatal conductance via Medlyn + profit-maximisation.

    Algorithm:

    1. Sweep gs from g0 to gs_max (30 points).
    
    2. At each gs, compute the full leaf physics chain:
       boundary layer → energy balance → photosynthesis → water potential.
    
    3. Compute normalised GAIN = An/An_max and COST = vulnerability(ψ_leaf).
    
    4. PROFIT = GAIN - COST. Find the gs that maximises profit.
    
    5. Clamp to Medlyn gs (hydraulics can only reduce gs, never increase it).
    
    6. Recompute final fluxes at the optimal gs.


    Parameters
    ----------
    physcon : PhysCon
    atmos : Atmos
    leaf : Leaf
        Must have: g0, g1_medlyn, minl_wp, a_psi.
    flux : Flux

    Returns
    -------
    flux : Flux
        With converged gs, An, Tleaf, psi_leaf, etc.
    """

    # Set initial values --------------------------------------------------------
    
    # Number of gs points in the sweep.
    n_points = 50

    # Vulnerability curve shape from leaf parameters
    a_psi = leaf.a_psi

    # Minimum conductance (e.g., 0.01)
    gs_min = leaf.g0

    # Practical upper limit
    gs_max = 0.8
    
    # Define arrays results -----------------------------------------------------
    
    # Create an array of evenly spaced numbers of size n_points that goes from
    # gs_min to gs_max
    gs_values = np.linspace(gs_min, gs_max, n_points)
    
    # Net photosynthesis (µmol CO₂/m²/s)
    an_arr = np.zeros(n_points)

    # Leaf water potential (MPa)
    psi_arr = np.zeros(n_points)

    # VPD at leaf surface (Pa)
    vpd_arr = np.zeros(n_points)

    # CO₂ at leaf surface (µmol/mol)
    cs_arr = np.zeros(n_points)
    
    # Profits array 
    profit_arr = np.zeros(n_points)
    
    # Boundary layer conductances (independent of gs, computed once) ------------
    flux = leaf_boundary_layer(physcon, atmos, leaf, flux)
    
    # Try many values of gs and compute full physics at each point --------------

    # At each gs, we compute:
    #   - Tleaf (energy balance)
    #   - An, cs, VPD (photosynthesis)
    #   - ψ_leaf (hydraulics — transpiration rate sets the water demand)
    for each_iteration, gs_trial in enumerate(gs_values):
        
        flux.gs = gs_trial

        # Energy balance: wider stomata → more transpiration → cooler leaf
        # Example: gs=0.05 → Tleaf≈35°C, gs=0.40 → Tleaf = 31°C
        flux = leaf_temperature(physcon, atmos, leaf, flux)

        # Photosynthesis: more CO₂ in (higher gs) → more carbon fixation
        # But An saturates because Rubisco has a maximum rate (Vcmax)
        flux = leaf_photosynthesis(physcon, atmos, leaf, flux)

        # Water potential: more transpiration pulls ψ_leaf more negative
        # This is the link between gas exchange and hydraulic risk
        flux = leaf_water_potential(physcon, leaf, flux)

        # Save results
        an_arr[each_iteration] = flux.an
        psi_arr[each_iteration] = flux.psi_leaf
        vpd_arr[each_iteration] = flux.vpd
        cs_arr[each_iteration] = flux.cs

    # Safety checks -------------------------------------------------------------

    # Get maximum of An. 
    an_max = np.max(an_arr)
    
    # Safety check: If no positive photosynthesis at any gs then dark or extreme 
    # stress. Set gs value to minimum conductance and compute everything again 
    # and STOP
    if an_max <= 0:
        
        flux.gs = leaf.g0
        flux = leaf_boundary_layer(physcon, atmos, leaf, flux)
        flux = leaf_temperature(physcon, atmos, leaf, flux)
        flux = leaf_photosynthesis(physcon, atmos, leaf, flux)
        flux = leaf_water_potential(physcon, leaf, flux)
        
        return flux

    # Safety hydraulic check: if even the lowest gs produces ψ_leaf below minl_wp,
    # the soil is so dry that NO stomatal opening is safe. Fall back to g0.
    # If the first calculation is below minl_wp then all the following are even 
    # worse, no need get the min(psi_arr) 
    if psi_arr[0] < leaf.minl_wp:
        
        # Even at minimum gs, ψ_leaf is below cavitation threshold.
        # Use minimum conductance
        flux.gs = leaf.g0
        flux = leaf_boundary_layer(physcon, atmos, leaf, flux)
        flux = leaf_temperature(physcon, atmos, leaf, flux)
        flux = leaf_photosynthesis(physcon, atmos, leaf, flux)
        flux = leaf_water_potential(physcon, leaf, flux)
        
        return flux

    # Compute gain, cost and profit ---------------------------------------------
    # If all safety checks passed then compute gain, cost and profit
    
    # Calculate profits
    for each_point in range(n_points):
        
        # GAIN: normalised carbon acquisition
        # Low gs: An is low → gain low (CO₂ starved)
        # High gs: An saturates → gain approaches 1.0
        gain = max(an_arr[each_point] / an_max, 0.0)

        # COST: normalised hydraulic risk
        # Low gs: ψ_leaf near zero → cost near zero (safe plumbing)
        # High gs: ψ_leaf very negative → cost approaches 1.0 (cavitation risk)
        cost = _hydraulic_cost(psi_arr[each_point], leaf.psi_50, a_psi)

        # PROFIT = GAIN − COST
        # Positive profit: the carbon gain outweighs the hydraulic risk
        # The peak of this curve is the optimal operating point
        profit_arr[each_point] = gain - cost

        # Hard hydraulic safety cap: if ψ_leaf is below the cavitation
        # threshold (minl_wp), set profit to -infinity so this gs is
        # never selected.

        # WHY: The sigmoidal vulnerability curve assigns finite cost even at
        # extremely negative ψ (e.g., cost=0.88 at ψ=-3 MPa), which can let gain
        # "outrun" cost at high gs under dry soil. The hard cap prevents this.
        # It says "regardless of the smooth cost function, absolute cavitation
        # is not allowed."

        # This combines the smooth early response of the profit framework
        # with the safety guarantee of a hard hydraulic limit.
        if psi_arr[each_point] < leaf.minl_wp:
            profit_arr[each_point] = -999.0

    # Find the gs that maximises profit -----------------------------------------
    # In other words find the gs values where GAIN - COST is largest. 
    # This is the peak of the profit curve in figure 4.6 where dF/dx = 0
    max_profit = np.argmax(profit_arr)
    gs_profit = gs_values[max_profit]

    # Compute the Medlyn gs at the profit-optimal conditions --------------------
    
    # The profit framework might suggest higher gs than Medlyn
    # if hydraulic cost is still low. I take min(profit_gs, medlyn_gs) so
    # that hydraulics can only REDUCE gs below Medlyn, never increase it.
    # Under well-watered conditions: gs = Medlyn (hydraulics don't bind)
    # Under drought: gs < Medlyn (hydraulics pull gs down smoothly)

    an_opt = an_arr[max_profit]
    cs_opt = cs_arr[max_profit]
    vpd_opt = vpd_arr[max_profit]

    # Convert Pa to kPa, floor at 0.1
    vpd_kpa = max(vpd_opt / 1000.0, 0.1)

    # Medlyn model
    if an_opt > 0 and cs_opt > 0:
        gs_medlyn = (
            leaf.g0 + 1.6 * (1.0 + leaf.g1_medlyn / np.sqrt(vpd_kpa)) * an_opt / cs_opt
        )
    else:
        gs_medlyn = leaf.g0

    # Take the more conservative of the two gs_profit vrs gs_medlyn
    gs_final = min(gs_profit, gs_medlyn)
    
    # Never below g0
    gs_final = max(gs_final, leaf.g0)  

    # Compute final fluxes at the optimal gs ------------------------------------
    flux.gs = gs_final
    flux = leaf_boundary_layer(physcon, atmos, leaf, flux)
    flux = leaf_temperature(physcon, atmos, leaf, flux)
    flux = leaf_photosynthesis(physcon, atmos, leaf, flux)
    flux = leaf_water_potential(physcon, leaf, flux)

    return flux